# A Fool Fraud — Dataset Analysis
**Exploratory Data Analysis (EDA) of the Credit Card Fraud Dataset**

This notebook provides a thorough analysis of the dataset before any modeling:
- Shape, types, and missing values
- Class imbalance visualization
- Feature distributions (fraud vs genuine)
- Correlation analysis
- Time and Amount patterns
- Outlier detection

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded.')

## 2. Load & Inspect Data

In [ ]:
# !wget https://storage.googleapis.com/qwasar-public/track-ds/creditcard.csv
df = pd.read_csv('creditcard.csv')

print(f'Shape       : {df.shape}')
print(f'Columns     : {df.shape[1]}')
print(f'Rows        : {df.shape[0]:,}')
print(f'Memory      : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 3. Class Distribution

In [ ]:
counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean() * 100

print(f'Genuine transactions : {counts[0]:,}  ({100 - fraud_rate:.4f}%)')
print(f'Fraudulent           : {counts[1]:,}  ({fraud_rate:.4f}%)')
print(f'Imbalance ratio      : {counts[0] / counts[1]:.0f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(['Genuine (0)', 'Fraud (1)'], counts.values, color=['#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Transaction Count by Class', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts.values,
    labels=[f'Genuine ({100-fraud_rate:.2f}%)', f'Fraud ({fraud_rate:.2f}%)'],
    colors=['#2196F3', '#F44336'],
    autopct='%1.3f%%',
    startangle=90,
    explode=(0, 0.1)
)
axes[1].set_title('Class Proportion', fontweight='bold')

plt.suptitle('Class Imbalance Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

## 4. Transaction Amount Analysis

In [ ]:
fraud   = df[df['Class'] == 1]
genuine = df[df['Class'] == 0]

print('Amount Statistics:')
print(pd.DataFrame({
    'Genuine': genuine['Amount'].describe(),
    'Fraud'  : fraud['Amount'].describe()
}).round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(genuine['Amount'], bins=80, alpha=0.6, color='#2196F3', label='Genuine', density=True)
axes[0].hist(fraud['Amount'],   bins=80, alpha=0.7, color='#F44336', label='Fraud',   density=True)
axes[0].set_xlim(0, 500)
axes[0].set_title('Transaction Amount Distribution (clipped at $500)', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Boxplot
df_amount = pd.DataFrame({'Amount': df['Amount'], 'Class': df['Class'].map({0: 'Genuine', 1: 'Fraud'})})
sns.boxplot(data=df_amount, x='Class', y='Amount', palette={'Genuine': '#2196F3', 'Fraud': '#F44336'}, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Amount Distribution (log scale)', fontweight='bold')
axes[1].set_ylabel('Amount ($) — log scale')

plt.suptitle('Transaction Amount by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('amount_analysis.png', bbox_inches='tight')
plt.show()

## 5. Time Analysis

In [ ]:
# Convert seconds to hours for readability
df['Hour'] = (df['Time'] / 3600).astype(int) % 24

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Transactions over time
axes[0].plot(genuine['Time'] / 3600, np.zeros(len(genuine)), '|', color='#2196F3', alpha=0.01, markersize=2)
axes[0].hist(genuine['Time'] / 3600, bins=96, alpha=0.6, color='#2196F3', label='Genuine', density=True)
axes[0].hist(fraud['Time'] / 3600,   bins=96, alpha=0.8, color='#F44336', label='Fraud',   density=True)
axes[0].set_xlabel('Hours since first transaction')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Frequency Over Time', fontweight='bold')
axes[0].legend()

# Fraud by hour of day
fraud_by_hour   = fraud.groupby('Hour').size()
genuine_by_hour = genuine.groupby('Hour').size()
fraud_rate_hour = (fraud_by_hour / (fraud_by_hour + genuine_by_hour) * 100).fillna(0)

axes[1].bar(fraud_rate_hour.index, fraud_rate_hour.values, color='#F44336', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('time_analysis.png', bbox_inches='tight')
plt.show()

## 6. Feature Distributions: Fraud vs Genuine

In [ ]:
pca_features = [f'V{i}' for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(20, 28))
axes = axes.ravel()

for i, feat in enumerate(pca_features):
    axes[i].hist(genuine[feat], bins=60, alpha=0.5, color='#2196F3', label='Genuine', density=True)
    axes[i].hist(fraud[feat],   bins=60, alpha=0.7, color='#F44336', label='Fraud',   density=True)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=7)

# Hide unused axes
for j in range(len(pca_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('PCA Feature Distributions: Genuine vs Fraud', fontsize=15, fontweight='bold', y=1.001)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()

## 7. Most Discriminative Features (by Mean Difference)

In [ ]:
mean_diff = abs(fraud[pca_features].mean() - genuine[pca_features].mean())
mean_diff = mean_diff.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
mean_diff.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Absolute Mean Difference (Fraud − Genuine) per Feature', fontweight='bold', fontsize=13)
ax.set_ylabel('|Mean Fraud − Mean Genuine|')
ax.set_xlabel('Feature')
ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('discriminative_features.png', bbox_inches='tight')
plt.show()

print('Top 5 most discriminative features:')
print(mean_diff.head())

## 8. Correlation Heatmap

In [ ]:
# Correlation with Class label
corr_with_class = df[pca_features + ['Amount', 'Class']].corr()['Class'].drop('Class').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar: correlation with Class
colors_corr = ['#F44336' if v < 0 else '#4CAF50' for v in corr_with_class.values]
axes[0].barh(corr_with_class.index, corr_with_class.values, color=colors_corr, edgecolor='black')
axes[0].axvline(x=0, color='black', lw=1)
axes[0].set_title('Feature Correlation with Class Label', fontweight='bold')
axes[0].set_xlabel('Pearson Correlation')
axes[0].grid(True, axis='x', alpha=0.3)

# Full correlation heatmap (PCA features only)
top_features = mean_diff.head(10).index.tolist() + ['Amount', 'Class']
corr_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, ax=axes[1], cbar_kws={'shrink': 0.8}
)
axes[1].set_title('Correlation Matrix (Top 10 Features + Amount)', fontweight='bold')

plt.tight_layout()
plt.savefig('correlation_analysis.png', bbox_inches='tight')
plt.show()

## 9. Outlier Detection

In [ ]:
# IQR-based outlier count per feature
outlier_counts = {}
for feat in pca_features + ['Amount']:
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[feat] < Q1 - 1.5 * IQR) | (df[feat] > Q3 + 1.5 * IQR)).sum()
    outlier_counts[feat] = outliers

outlier_series = pd.Series(outlier_counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
outlier_series.plot(kind='bar', ax=ax, color='darkorange', edgecolor='black')
ax.set_title('Outlier Count per Feature (IQR Method)', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Outliers')
ax.grid(True, axis='y', alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outliers.png', bbox_inches='tight')
plt.show()

print('Features with most outliers:')
print(outlier_series.head(10))

## 10. Summary & Key Findings

In [ ]:
print('=' * 55)
print('  DATASET ANALYSIS — KEY FINDINGS')
print('=' * 55)
print(f'  Total transactions   : {len(df):,}')
print(f'  Fraudulent           : {df["Class"].sum():,} ({df["Class"].mean()*100:.4f}%)')
print(f'  Imbalance ratio      : {counts[0]/counts[1]:.0f}:1')
print(f'  Features             : {len(pca_features)} PCA + Amount + Time')
print(f'  Missing values       : None')
print()
print('  Top 3 discriminative features:')
for feat, val in mean_diff.head(3).items():
    print(f'    {feat}: mean diff = {val:.4f}')
print()
print('  Challenges for modeling:')
print('    - Severe class imbalance requires special handling')
print('    - Accuracy is a misleading metric — use F1 / ROC-AUC')
print('    - Amount and Time need scaling (PCA features do not)')
print('    - Some features have significant outliers')
print('=' * 55)